# Phase 1.1：文件、文本和 Document 数据结构

## 目标

从一个真实 Markdown 文件开始，不调用解析器，先自己观察：路径如何指向文件，文本如何读进内存，为什么需要 `source/page/metadata`。

**本课交付：** `data/processed/document_inventory.json`。

## Evidence Quest 任务卡：Phase 1.1：文档考古现场

**你的身份：** 文档考古员  
**案件背景：** 你在现场找到了一份没有标签的 Markdown 案件材料。搜索系统不能只拿到正文，还必须知道证据来自哪个文件。

### 本关专业 Goal

把原始文件变成带 source、page 和 metadata 的第一份 Document。

### 你要交付的作品

**文档清单 + 可追溯 Document 记录**

### 通关判定

- 先运行带逐行中文注释的示范，预测输出，再自己重新敲一遍关键代码。
- 至少改变一个参数或输入，记录它为什么改变了结果。
- 完成末尾的 Boss Challenge，并能解释一个失败样本。
- 把本关产物交给下一关，而不是把代码停留在 Notebook 屏幕上。

**通关奖励：** 解锁徽章：来源追踪员  
**学习节奏：** 看故事 -> 跟敲一小段 -> 观察输出 -> 自己改写 -> 验收作品。

## 先建立三个词的区别

- **Path**：文件在磁盘上的位置。
- **Text**：文件内容被读进 Python 后的字符串。
- **Document**：Text 加上来源、页码和格式等描述信息。

后面的 Chunk 是从 Document 切出来的更小单位。把这些层次分开，才能知道问题发生在“读文件”还是“切文本”。

In [1]:
# 导入 Path，用它表示跨平台的文件路径。
from pathlib import Path

# 导入 json，用它读取和保存项目的结构化数据。
import json

# 导入 sys，用它把项目根目录加入 Python 的模块搜索路径。
import sys


# 定义一个函数，负责从当前工作目录向上查找项目根目录。
def find_project_root() -> Path:
    # 把当前目录和它的所有父目录放进候选列表。
    candidates = [Path.cwd(), *Path.cwd().parents]

    # 逐个检查候选目录是否包含本项目的两个核心模块目录。
    for candidate in candidates:
        # 找到同时存在的目录时，返回这个候选目录。
        if (candidate / "phase1_doc_parser").is_dir() and (candidate / "phase2_semantic_search").is_dir():
            return candidate

    # 如果所有候选目录都不符合，说明 Jupyter 启动位置不在项目内。
    raise RuntimeError("找不到项目根目录，请从 ai-search-rag-internship 启动 JupyterLab")


# 执行查找函数，得到当前项目根目录。
ROOT = find_project_root()

# 如果项目根目录还不在模块搜索路径中，就把它添加进去。
if str(ROOT) not in sys.path:
    # 把项目根目录插入最前面，确保导入的是当前项目代码。
    sys.path.insert(0, str(ROOT))

# 打印根目录，帮助学习者确认 Notebook 没有在错误目录运行。
print("项目根目录:", ROOT)

项目根目录: D:\code\codeByCursor\AI_EXAM\ai-search-rag-internship


In [2]:
# 定义本关任务编号，后面的记录会用它区分不同阶段。
QUEST_STAGE = 'phase1.1'

# 定义学习者可以持续保存的案件档案路径。
QUEST_PROFILE_PATH = ROOT / "data" / "processed" / "evidence_quest_profile.json"

# 如果第一次打开课程还没有档案，就使用一个安全的默认案件。
default_profile = {
    "case_name": "校园知识库失踪案",
    "audience": "需要快速查证资料的同学",
    "must_answer": "证据来自哪里，能否回到原文？",
    "must_refuse": "检索结果没有证据时必须说不知道",
    "xp": 0,
    "badges": [],
}

# 检查任务档案是否已经由 Mission Control 创建。
if QUEST_PROFILE_PATH.is_file():
    # 读取学员自己的案件主题，让所有 Notebook 共享同一个故事。
    quest_profile = json.loads(QUEST_PROFILE_PATH.read_text(encoding="utf-8"))
else:
    # 没有档案时复制默认值，避免直接修改模板字典。
    quest_profile = dict(default_profile)

# 计算当前累计经验值；错误值按 0 处理，避免看板阻塞学习。
quest_xp = int(quest_profile.get("xp", 0))

# 读取已经获得的徽章，并复制成当前 Notebook 的列表。
quest_badges = list(quest_profile.get("badges", []))

# 用可见的文字看板告诉学习者自己正在解决哪个真实问题。
print("Evidence Quest / 当前关卡:", QUEST_STAGE)
print("案件:", quest_profile.get("case_name", default_profile["case_name"]))
print("服务对象:", quest_profile.get("audience", default_profile["audience"]))
print("累计 XP:", quest_xp, "| 徽章:", ", ".join(quest_badges) if quest_badges else "尚未获得")

Evidence Quest / 当前关卡: phase1.1
案件: 校园知识库失踪案
服务对象: 需要复习课程资料的同学
累计 XP: 0 | 徽章: 案件接收员


In [3]:
# 指向项目中的 Markdown 教学文件。
markdown_path = ROOT / "phase1_doc_parser" / "examples" / "input" / "quickstart.md"

# 确认路径确实指向一个文件，而不是拼错的路径。
assert markdown_path.is_file()

# 打印路径的字符串形式，理解 Path 对象和普通字符串的关系。
print("文件路径:", markdown_path)

# 打印扩展名，后续可以用它选择不同解析器。
print("文件扩展名:", markdown_path.suffix)

文件路径: D:\code\codeByCursor\AI_EXAM\ai-search-rag-internship\phase1_doc_parser\examples\input\quickstart.md
文件扩展名: .md


### 逐句理解 `Path`

`ROOT / "phase1_doc_parser"` 不是字符串拼接，而是 Path 的路径连接操作。它会根据操作系统选择正确的分隔符。`is_file()` 是在真正读取之前做的边界检查，错误会更早、更容易理解。

In [4]:
# 以 UTF-8 编码读取整个 Markdown 文件。
raw_text = markdown_path.read_text(encoding="utf-8-sig")

# 打印原始字符数，观察文件内容进入内存后的规模。
print("原始字符数:", len(raw_text))

# 打印前 200 个字符，确认读取到的是正文而不是二进制乱码。
print(raw_text[:200])

# 去除首尾空白，得到后续解析使用的正文。
normalized_text = raw_text.strip()

# 验证正文没有因为清理而变成空字符串。
assert normalized_text

原始字符数: 140
# Phase 1 Quickstart

文档解析的目标不是尽快删除格式信息，而是保留足够的来源元数据，让后续检索结果可以回溯到原文。

## Chunk 策略

先按段落和换行切分，再按中文标点递归降级。overlap 用于保留跨边界的上下文，但会增加索引体积和重复召回。




## 1. 自己构造第一条 Document

现在先不用 `dataclass`，用普通字典表达一份文档。这样你能直接看到每个字段是什么；下一步再对比生产模块的 `ParsedDocument`。

In [5]:
# 检查本单元格依赖的前置变量是否已经创建。
required_variables = {"markdown_path", "normalized_text"}

# 找出当前 Kernel 中尚未存在的前置变量。
missing_variables = sorted(name for name in required_variables if name not in globals())

# 如果学习者跳过了前面的单元格，就给出可执行的修复提示。
if missing_variables:
    raise RuntimeError("请先从本 Notebook 顶部依次运行前面的代码单元格；缺少变量: " + ", ".join(missing_variables))

# 创建一个最小 Document 字典，把正文放到 text 字段。
manual_document = {"text": normalized_text}

# 加入 source，让未来的检索结果可以回到原始文件。
manual_document["source"] = str(markdown_path)

# Markdown 没有 PDF 页码，因此用 None 表示页码不存在。
manual_document["page"] = None

# 加入格式信息，让下游知道这段内容来自 Markdown。
manual_document["metadata"] = {"format": "markdown"}

# 打印手工构造的数据，观察字段和值的对应关系。
print(json.dumps(manual_document, ensure_ascii=False, indent=2))

# 检查四个基础字段都存在。
assert {"text", "source", "page", "metadata"} <= manual_document.keys()

{
  "text": "# Phase 1 Quickstart\n\n文档解析的目标不是尽快删除格式信息，而是保留足够的来源元数据，让后续检索结果可以回溯到原文。\n\n## Chunk 策略\n\n先按段落和换行切分，再按中文标点递归降级。overlap 用于保留跨边界的上下文，但会增加索引体积和重复召回。",
  "source": "D:\\code\\codeByCursor\\AI_EXAM\\ai-search-rag-internship\\phase1_doc_parser\\examples\\input\\quickstart.md",
  "page": null,
  "metadata": {
    "format": "markdown"
  }
}


### 为什么不只保存字符串？

如果只保存 `text`，检索命中后用户看不到来源；如果只保存 `source`，系统没有可供匹配的正文；如果把页码藏在文件名里，PDF 页码和 Markdown 会变得不一致。稳定字段是后续阶段之间的契约。

In [6]:
# 导入生产解析函数，用相同文件生成正式 Document 对象。
from phase1_doc_parser.parser import parse_file

# 运行生产解析器，观察它返回的统一结构。
parsed_document = parse_file(markdown_path)[0]

# 打印生产对象，和上面的字典进行逐字段对比。
print(parsed_document)

# 检查生产解析器保留了同样的正文和来源信息。
assert parsed_document.text == normalized_text
assert parsed_document.source == str(markdown_path)
assert parsed_document.page is None

ParsedDocument(text='# Phase 1 Quickstart\n\n文档解析的目标不是尽快删除格式信息，而是保留足够的来源元数据，让后续检索结果可以回溯到原文。\n\n## Chunk 策略\n\n先按段落和换行切分，再按中文标点递归降级。overlap 用于保留跨边界的上下文，但会增加索引体积和重复召回。', source='D:\\code\\codeByCursor\\AI_EXAM\\ai-search-rag-internship\\phase1_doc_parser\\examples\\input\\quickstart.md', page=None, metadata={'format': 'markdown', 'headings': ['Phase 1 Quickstart', 'Chunk 策略']})


## 2. 处理格式差异：Markdown heading 是有价值的元数据

Markdown 的 `# 标题` 不只是正文字符，它可以帮助我们解释 Chunk 所属章节。生产解析器用正则提取标题并存入 `metadata["headings"]`。这里先用一个最小循环观察“逐行处理”是什么。

In [7]:
# 创建一个空列表，准备保存以 # 开头的标题文本。
heading_lines = []

# 按换行把正文拆成一行一行的字符串。
for line in normalized_text.splitlines():
    # 去除行首尾空白，避免标题结果带有多余空格。
    cleaned_line = line.strip()

    # 只把 Markdown 一级到六级标题识别为 heading。
    if cleaned_line.startswith("#") and not cleaned_line.startswith("######"):
        # 去掉标题符号和左侧空白，只保留标题文字。
        heading_lines.append(cleaned_line.lstrip("#").strip())

# 输出手工提取的标题，理解元数据如何从正文派生。
print("手工提取的标题:", heading_lines)

# 生产解析器应至少返回相同的标题信息。
assert parsed_document.metadata["headings"]

手工提取的标题: ['Phase 1 Quickstart', 'Chunk 策略']


## 3. 生成文档清单

真实项目通常不是只读一个文件。批处理前先生成清单，可以提前发现空文件、未知扩展名和文件规模异常。这个清单不是最终 Chunk，而是解析阶段的可审计输入记录。

In [8]:
# 指向所有原始输入文件所在目录。
input_directory = ROOT / "phase1_doc_parser" / "examples" / "input"

# 找到目录下的全部普通文件并排序。
input_files = sorted(path for path in input_directory.iterdir() if path.is_file())

# 创建空列表，用于保存每个文件的检查结果。
inventory = []

# 逐个检查输入文件。
for path in input_files:
    # 读取文件字节大小，发现明显的空文件。
    byte_size = path.stat().st_size

    # 记录文件名、扩展名、大小和是否被支持。
    inventory.append({"name": path.name, "suffix": path.suffix.lower(), "bytes": byte_size, "supported": path.suffix.lower() in {".md", ".markdown", ".txt", ".pdf"}})

# 创建数据目录，确保清单有固定存放位置。
output_directory = ROOT / "data" / "processed"
output_directory.mkdir(parents=True, exist_ok=True)

# 指定清单文件路径。
inventory_path = output_directory / "document_inventory.json"

# 写入 UTF-8 JSON，保留缩进方便检查。
inventory_path.write_text(json.dumps(inventory, ensure_ascii=False, indent=2), encoding="utf-8")

# 打印清单路径和文件数量。
print("已生成:", inventory_path, "files=", len(inventory))

# 确保清单覆盖了每一个输入文件。
assert len(inventory) == len(input_files)

已生成: D:\code\codeByCursor\AI_EXAM\ai-search-rag-internship\data\processed\document_inventory.json files= 2


## 本课验收

- [ ] 能解释 Path、Text、Document 的区别。
- [ ] 能手工构造一条包含 `text/source/page/metadata` 的 Document。
- [ ] 能说出为什么 Markdown 标题属于有用元数据。
- [ ] 已生成 `document_inventory.json`。

下一课不再把整篇文档当成一个字符串，而是从最简单的字符串切片开始写 Chunking。

## Boss Challenge：不看上面的示范，重新构造一个包含 text、source、page、metadata 的 Document。

下面是**故意保持注释状态**的跟敲模板。请先自己写，再取消注释逐行运行；不要把它当成需要复制的答案。

In [9]:
# 第 1 行：先写出本挑战需要的新变量或新输入。
# challenge_input = ...

# 第 2 行：调用本课已经学会的函数或模块。
# challenge_result = ...

# 第 3 行：打印一个中间结果，先观察再下结论。
# print(challenge_result)

# 第 4 行：写一个断言，把你的理解变成机器可检查的条件。
# assert ...

## 作品检查站

作品不是‘我运行过代码’，而是别人可以在文件浏览器中找到、下一阶段可以读取、你能解释生成过程的证据。下面的检查只报告事实，不替你假装通关。

In [10]:
# 列出本关应该产生的作品路径。
quest_artifact_candidates = ['data/processed/document_inventory.json']

# 把相对路径转换为项目根目录下的绝对路径。
quest_artifact_paths = [ROOT / path for path in quest_artifact_candidates]

# 只保留已经真正写入磁盘的作品。
quest_existing_artifacts = [str(path.relative_to(ROOT)) for path in quest_artifact_paths if path.is_file()]

# 保存一个不依赖外部服务的本关检查结果，方便复盘。
quest_checkpoint = {"stage": QUEST_STAGE, "existing_artifacts": quest_existing_artifacts}

# 打印检查结果，让学习者知道下一步是继续学习还是补交作品。
print("本关作品:", quest_existing_artifacts if quest_existing_artifacts else "还没有生成，请回到交付单元格")

本关作品: ['data\\processed\\document_inventory.json']
